# Model Evaluation: Job Category Classifier

**Goal:** Compare Logistic Regression and Random Forest baselines on the held-out test set, visualize per-class performance, and document the model selection rationale.

**Data source:** Pre-computed evaluation artifacts generated by `src/models/evaluate_model.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Paths
MODELS_DIR = Path('../models')
MODELS = {
    'Logistic Regression': MODELS_DIR / 'logistic_regression' / 'evaluation',
    'Random Forest': MODELS_DIR / 'random_forest' / 'evaluation',
}

## 1. Load Evaluation Artifacts

In [ ]:
def load_report(eval_dir: Path) -> pd.DataFrame:
    """Load classification report JSON as a DataFrame."""
    with open(eval_dir / 'classification_report.json') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    # Keep only per-class rows (drop accuracy, macro avg, weighted avg)
    aggregate_rows = ['accuracy', 'macro avg', 'weighted avg']
    classes = [idx for idx in df.index if idx not in aggregate_rows]
    return df.loc[classes], df


def load_confusion(eval_dir: Path) -> pd.DataFrame:
    """Load confusion matrix CSV."""
    return pd.read_csv(eval_dir / 'confusion_matrix.csv', index_col=0)


# Load all artifacts
reports = {}
reports_full = {}
confusions = {}

for name, eval_dir in MODELS.items():
    reports[name], reports_full[name] = load_report(eval_dir)
    confusions[name] = load_confusion(eval_dir)
    print(f'{name}: {len(reports[name])} classes loaded')

## 2. Aggregate Metrics Comparison

In [ ]:
# Build comparison table from the aggregate rows
rows = []
for name, df_full in reports_full.items():
    rows.append({
        'Model': name,
        'F1 Macro': df_full.loc['macro avg', 'f1-score'],
        'Precision Macro': df_full.loc['macro avg', 'precision'],
        'Recall Macro': df_full.loc['macro avg', 'recall'],
        'Accuracy': df_full.loc['accuracy', 'precision'],
        'F1 Weighted': df_full.loc['weighted avg', 'f1-score'],
    })

comparison = pd.DataFrame(rows).set_index('Model')
comparison.style.format('{:.4f}').highlight_max(axis=0, color='#d4edda')

## 3. Per-Class F1 Score Comparison

In [ ]:
f1_comparison = pd.DataFrame({
    name: df['f1-score'] for name, df in reports.items()
})

fig, ax = plt.subplots(figsize=(12, 6))
f1_comparison.plot.barh(ax=ax, width=0.75)
ax.set_xlabel('F1 Score')
ax.set_title('Per-Class F1 Score: LR vs RF')
ax.axvline(x=0.70, color='red', linestyle='--', alpha=0.5, label='Target (0.70)')
ax.legend(loc='lower right')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. RF Improvement Over LR

In [ ]:
delta = f1_comparison['Random Forest'] - f1_comparison['Logistic Regression']
delta = delta.sort_values()

colors = ['#e74c3c' if v < 0 else '#27ae60' for v in delta]

fig, ax = plt.subplots(figsize=(10, 5))
delta.plot.barh(ax=ax, color=colors)
ax.set_xlabel('F1 Difference (RF - LR)')
ax.set_title('Random Forest Improvement Over Logistic Regression')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 5. Confusion Matrices (Normalized)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, (name, cm) in zip(axes, confusions.items()):
    # Normalize by row (true label)
    cm_norm = cm.div(cm.sum(axis=1), axis=0)
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        ax=ax,
        cbar=False,
        linewidths=0.5,
        vmin=0,
        vmax=1,
    )
    ax.set_title(f'{name}', fontsize=14)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.suptitle('Normalized Confusion Matrices (row = recall)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 6. Weakest Classes (RF)

In [ ]:
rf_report = reports['Random Forest'].copy()
rf_report = rf_report.sort_values('f1-score')

print('Classes sorted by F1 (ascending):\n')
rf_report[['f1-score', 'precision', 'recall', 'support']].head(5).style.format({
    'f1-score': '{:.3f}',
    'precision': '{:.3f}',
    'recall': '{:.3f}',
    'support': '{:.0f}',
})

## 7. Summary

**Selected model: Random Forest.**

Random Forest is selected because it achieves the highest macro F1 on the held-out test set. Both baselines exceed the target threshold of 0.70 macro F1, so the classification objective is met.

Logistic Regression remains useful as an interpretable baseline, especially after simplifying the feature pipeline to text-only TF-IDF. Random Forest provides better final predictive performance, at the cost of lower interpretability.

The weakest areas remain classes with low support or ambiguous boundaries, especially Backend and Software Engineer.